# Library 01: loading the Cricsheet files and choosing the matches

Functions for **Stage 1** (the list of source files) and **Stage 2** (the match filters).

How we handle the raw data:
* The Cricsheet zip file is our only data source and we never change it. We unzip it once into
  `data/raw/england_male_json/`.
* Every JSON file gets a line in the source manifest with its size and SHA-256 checksum, so we can always tell
  if a file has changed.
* Every match is either kept or left out with a written reason. No file is dropped without a trace.

Needs `lib_00_settings`.

In [ ]:
def file_checksum(file_path: Path, block_size: int = 1 << 20) -> str:
    """SHA-256 checksum of a file. We read it in 1 MB blocks so even the big zip file uses little memory."""
    checksum = hashlib.sha256()
    with file_path.open("rb") as open_file:
        while block := open_file.read(block_size):
            checksum.update(block)
    return checksum.hexdigest()


def extract_archive_if_needed(archive_path: Path = RAW_ARCHIVE_PATH, match_folder: Path = RAW_MATCH_FOLDER) -> Path:
    """Unzip the Cricsheet archive, but only if the match files are not there yet. Returns the match folder."""
    already_extracted = match_folder.exists() and any(match_folder.glob("*.json"))
    if not already_extracted:
        # Edge case: someone cloned the project without the zip file. We stop with a clear message
        # saying where to get it, instead of failing later with a confusing error.
        if not archive_path.exists():
            raise FileNotFoundError(f"{archive_path} is missing. Download it from {ARCHIVE_SOURCE_URL} into data/raw/.")
        match_folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(match_folder)
    return match_folder


def list_match_files(match_folder: Path = RAW_MATCH_FOLDER) -> list[Path]:
    """All match files, sorted by name. Sorting matters: it makes every run read the files in the same order."""
    return sorted(match_folder.glob("*.json"))


def load_match_file(match_path: Path) -> dict[str, Any]:
    """Read one match file into a Python dictionary."""
    with match_path.open(encoding="utf-8") as open_file:
        return json.load(open_file)

## Source manifest

One row per file. If a file can't be read, it still gets a row, with `readable = False`, so that we notice it.

In [ ]:
def build_source_manifest(match_folder: Path = RAW_MATCH_FOLDER) -> pd.DataFrame:
    """List every source file with its size, checksum and Cricsheet schema version."""
    manifest_rows = []
    for match_path in list_match_files(match_folder):
        manifest_row = {
            "filename": match_path.name,
            "bytes": match_path.stat().st_size,
            "sha256": file_checksum(match_path),
            "readable": True,
            "data_version": "",
            "created": "",
            "revision": -1,
        }
        try:
            file_meta = load_match_file(match_path).get("meta", {})
            manifest_row["data_version"] = str(file_meta.get("data_version", ""))
            manifest_row["created"] = str(file_meta.get("created", ""))
            manifest_row["revision"] = int(file_meta.get("revision", -1))
        except (json.JSONDecodeError, OSError, ValueError):
            # Edge case: a broken or half-downloaded file. We keep it in the list and mark it.
            manifest_row["readable"] = False
        manifest_rows.append(manifest_row)
    return pd.DataFrame(manifest_rows)

## Match details and filters

`describe_match` pulls out the match details we need into one flat row.
`find_exclusion_reason` then checks the filters **always in the same order** and returns the first one that
fails. A match that passes everything gets an empty string, which means "keep it".

In [ ]:
def to_date(value: Any) -> date:
    """Turn a Cricsheet date such as '2022-06-01' into a date object (dates that are already dates pass through)."""
    return value if isinstance(value, date) else date.fromisoformat(str(value))


def describe_match(match_data: dict[str, Any], filename: str) -> dict[str, Any]:
    """Collect the match details we use (dates, teams, format, result, number of innings) into one row."""
    match_info = match_data["info"]
    match_days = [to_date(day) for day in match_info.get("dates", [])]
    team_names = list(match_info.get("teams", []))
    match_outcome = match_info.get("outcome", {})
    # Edge case: some files have "event": null instead of leaving it out, so we fall back to an empty dict.
    match_event = match_info.get("event", {}) or {}
    all_innings = match_data.get("innings", [])
    return {
        "match_id": filename.rsplit(".", 1)[0],
        "filename": filename,
        # Test matches have several dates; the first one is the start of the match.
        "start_date": min(match_days) if match_days else None,
        "end_date": max(match_days) if match_days else None,
        "gender": match_info.get("gender"),
        "team_type": match_info.get("team_type"),
        "match_type": match_info.get("match_type"),
        "format": CRICSHEET_TO_FORMAT.get(match_info.get("match_type", ""), ""),
        "team_1": team_names[0] if len(team_names) > 0 else "",
        "team_2": team_names[1] if len(team_names) > 1 else "",
        "opponent": next((team for team in team_names if team != TEAM_NAME), ""),
        "event": match_event.get("name", ""),
        "venue": match_info.get("venue", ""),
        "outcome_winner": match_outcome.get("winner", ""),
        "outcome_result": match_outcome.get("result", ""),
        "outcome_method": match_outcome.get("method", ""),
        "n_innings_total": len(all_innings),
        # Edge case: a match abandoned before England batted has 0 here. It passes the filters but gives no partnerships.
        "n_innings_england": sum(1 for innings in all_innings if innings.get("team") == TEAM_NAME),
    }


def find_exclusion_reason(match_record: dict[str, Any]) -> str:
    """Return '' if the match belongs to the study, otherwise the name of the first filter it fails."""
    if match_record["gender"] != TEAM_GENDER:
        return "gender_not_male"
    if match_record["team_type"] != TEAM_TYPE:
        return "not_international"
    if TEAM_NAME not in (match_record["team_1"], match_record["team_2"]):
        return "england_not_playing"
    if not match_record["format"]:
        return "format_not_target"
    # Edge case: a file without any date can't be placed in the window, so it gets its own reason.
    if match_record["start_date"] is None:
        return "missing_date"
    if match_record["start_date"] < STUDY_START_DATE:
        return "before_window"
    if match_record["start_date"] > STUDY_END_DATE:
        return "after_window"
    return ""


def audit_match_files(match_paths: Iterable[Path]) -> pd.DataFrame:
    """Describe every file and mark it as included or not (with the reason)."""
    audit_rows = []
    for match_path in match_paths:
        try:
            match_record = describe_match(load_match_file(match_path), match_path.name)
            exclusion_reason = find_exclusion_reason(match_record)
        except Exception as error:
            # Edge case: a file we can't parse. We record why instead of skipping it quietly.
            match_record = {"match_id": match_path.stem, "filename": match_path.name}
            exclusion_reason = f"unreadable:{type(error).__name__}"
        match_record["included"] = exclusion_reason == ""
        match_record["exclusion_reason"] = exclusion_reason
        audit_rows.append(match_record)
    return pd.DataFrame(audit_rows)